<a href="https://colab.research.google.com/github/ReevaKanakhara/Burn-Scar-Delineation-Similipal/blob/main/Copy_of_Similipal(Attention_U_NET).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q geopandas rasterio tensorflow matplotlib scikit-learn scikit-image

import numpy as np
import rasterio
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
from google.colab import drive
from skimage.util import view_as_windows
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

#Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Imports successful!")

In [ ]:
# STEP 1: MOUNT GOOGLE DRIVE
try:
    drive.mount('/content/drive', force_remount=False)
    BASE_PATH = '/content/drive/MyDrive/P7_Simlipal/'
    print(f"Drive mounted: {BASE_PATH}")
except Exception as e:
    print(f"Drive mount failed: {e}")
    raise

In [ ]:
 # STEP 2: LOAD SATELLITE IMAGERY WITH VALIDATION
before_path = BASE_PATH + 'Sentinel2_FebMar2021_Before.tif'
after_path  = BASE_PATH + 'Sentinel2_AprMay2021_After.tif'

def safe_read_band(src, band_num, name="band"):
    """Read band with full NaN protection"""
    try:
        data = src.read(band_num).astype(np.float32)
        #Replace NaN, inf, and invalid values
        data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
        #Clip to Sentinel-2 range
        data = np.clip(data, 0, 20000)
        print(f"{name}: shape={data.shape}, range=[{data.min():.0f}, {data.max():.0f}]")
        return data
    except Exception as e:
        print(f"Failed to read {name}: {e}")
        raise

print("Loading BEFORE image (Feb-Mar 2021)...")
with rasterio.open(before_path) as src_b:
    nir_before = safe_read_band(src_b, 4, "NIR")
    swir2_before = safe_read_band(src_b, 5, "SWIR2")

print("Loading AFTER image (Apr-May 2021)...")
with rasterio.open(after_path) as src_a:
    r_after = safe_read_band(src_a, 3, "Red")
    g_after = safe_read_band(src_a, 2, "Green")
    b_after = safe_read_band(src_a, 1, "Blue")
    nir_after = safe_read_band(src_a, 4, "NIR")
    swir2_after = safe_read_band(src_a, 5, "SWIR2")

print("All imagery loaded successfully.")


In [ ]:
# STEP 3: COMPUTE BURN SEVERITY(dNBR)
def safe_nbr(nir, swir2, epsilon=1e-8):
    """
    Compute Normalized Burn Ratio with comprehensive NaN protection
    NBR = (NIR - SWIR2) / (NIR + SWIR2)
    """
    # Force valid range (typical Sentinel-2 reflectance: 0-10000)
    nir = np.clip(np.nan_to_num(nir, nan=1000.0), 1.0, 15000.0)
    swir2 = np.clip(np.nan_to_num(swir2, nan=500.0), 1.0, 15000.0)

    # Safe division with epsilon
    numerator = nir - swir2
    denominator = nir + swir2 + epsilon

    nbr = numerator / denominator

    # Clip to valid NBR range [-1, 1]
    nbr = np.clip(nbr, -1.0, 1.0)

    # Final NaN check
    nbr = np.nan_to_num(nbr, nan=0.0)

    return nbr

print("Computing burn indices...")
nbr_before = safe_nbr(nir_before, swir2_before)
nbr_after = safe_nbr(nir_after, swir2_after)

#dNBR (higher values = more severe burn)
dNBR = nbr_before - nbr_after
dNBR = np.nan_to_num(dNBR, nan=0.0)

#Create binary burn mask (threshold: dNBR > 0.1)
burned_mask = (dNBR > 0.1).astype(np.float32)

print(f"dNBR range: [{dNBR.min():.3f}, {dNBR.max():.3f}]")
print(f"Burned pixels: {np.sum(burned_mask):,} ({100*np.mean(burned_mask):.2f}%)")
print("Burn severity computed")

In [ ]:
# STEP 4: DATASET PREPARATION - PATCH EXTRACTION
print("Preparing training dataset...")

PATCH_SIZE = 64
STRIDE = 32

# Get dimensions
h, w = r_after.shape
print(f"Image size: {h} × {w}")

# Create RGB composite (normalized to [0, 1])
rgb = np.stack([r_after, g_after, b_after], axis=-1)
rgb = np.clip(rgb / 3000.0, 0, 1).astype(np.float32)

# Ensure same dimensions
mask = burned_mask.astype(np.float32)

# Pad to ensure full coverage
pad_h = (PATCH_SIZE - h % STRIDE) % STRIDE
pad_w = (PATCH_SIZE - w % STRIDE) % STRIDE

rgb_padded = np.pad(rgb, ((0, pad_h), (0, pad_w), (0, 0)), mode='reflect')
mask_padded = np.pad(mask, ((0, pad_h), (0, pad_w)), mode='reflect')

print(f"Padded size: {rgb_padded.shape[0]} × {rgb_padded.shape[1]}")

# Extract patches using sliding window
try:
    X_patches = view_as_windows(
        rgb_padded,
        (PATCH_SIZE, PATCH_SIZE, 3),
        step=STRIDE
    )
    y_patches = view_as_windows(
        mask_padded,
        (PATCH_SIZE, PATCH_SIZE),
        step=STRIDE
    )

    # Reshape to (num_patches, height, width, channels)
    n_h, n_w = X_patches.shape[0], X_patches.shape[1]
    X = X_patches.reshape(-1, PATCH_SIZE, PATCH_SIZE, 3)
    y = y_patches.reshape(-1, PATCH_SIZE, PATCH_SIZE, 1)

    print(f"Raw patches: {len(X)}")

except Exception as e:
    print(f"Patch extraction failed: {e}")
    raise

#Remove any patches with NaN
valid_X = np.isfinite(X).all(axis=(1, 2, 3))
valid_y = np.isfinite(y).all(axis=(1, 2, 3))
valid_mask = valid_X & valid_y

X = X[valid_mask]
y = y[valid_mask]

print(f"Valid patches: {len(X)}")
assert len(X) > 0, "No valid patches found!"

# STEP 5:PREVENT CLASS IMBALANCE

# Calculate burn ratio per patch
burn_ratios = np.mean(y, axis=(1, 2, 3))
burned_patches = burn_ratios > 0.01  # At least 1% burned

n_burned = np.sum(burned_patches)
n_total = len(y)

print(f"Balancing dataset...")
print(f"Burned patches: {n_burned} ({100*n_burned/n_total:.1f}%)")

# Keep all burned patches + 3x non-burned patches
if n_burned > 0:
    n_nonburned_keep = min(n_burned * 3, n_total - n_burned)

    burned_idx = np.where(burned_patches)[0]
    nonburned_idx = np.where(~burned_patches)[0]

    if len(nonburned_idx) > 0:
        selected_nonburned = np.random.choice(
            nonburned_idx,
            min(n_nonburned_keep, len(nonburned_idx)),
            replace=False
        )
        selected_idx = np.concatenate([burned_idx, selected_nonburned])
    else:
        selected_idx = burned_idx

    np.random.shuffle(selected_idx)
    X = X[selected_idx]
    y = y[selected_idx]

    print(f"Balanced dataset: {len(X)} patches")
    print(f"New burn ratio: {100*np.mean(y):.1f}%")

# STEP 6: TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(f"Dataset split:")
print(f"Training: {len(X_train)} patches (burn: {100*np.mean(y_train):.1f}%)")
print(f"Testing:  {len(X_test)} patches (burn: {100*np.mean(y_test):.1f}%)")

# Final validation
assert np.isfinite(X_train).all(), "Training data contains NaN/inf!"
assert np.isfinite(y_train).all(), "Training labels contain NaN/inf!"
print("Dataset ready")

# STEP 7: BUILD ATTENTION U-NET MODEL

class AttentionGate(layers.Layer):
    """
    Attention Gate for U-Net
    Helps the model focus on relevant features during upsampling
    """
    def __init__(self, filters, **kwargs):
        super(AttentionGate, self).__init__(**kwargs)
        self.filters = filters

    def build(self, input_shape):
        # Input shape: [(batch, h, w, c_g), (batch, h, w, c_x)]
        self.W_g = layers.Conv2D(self.filters, 1, padding='same', use_bias=True)
        self.W_x = layers.Conv2D(self.filters, 1, padding='same', use_bias=True)
        self.psi = layers.Conv2D(1, 1, padding='same', use_bias=True)
        self.relu = layers.ReLU()
        self.sigmoid = layers.Activation('sigmoid')
        super(AttentionGate, self).build(input_shape)

    def call(self, inputs):
        g, x = inputs  # g: gating signal, x: skip connection

        # Apply convolutions
        g1 = self.W_g(g)
        x1 = self.W_x(x)

        # Add and activate
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        psi = self.sigmoid(psi)

        # Apply attention
        return x * psi

    def get_config(self):
        config = super(AttentionGate, self).get_config()
        config.update({'filters': self.filters})
        return config


def conv_block(x, filters, kernel_size=3, name_prefix='conv'):
    """Convolutional block with BatchNorm"""
    x = layers.Conv2D(filters, kernel_size, padding='same',
                     activation='relu', name=f'{name_prefix}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn1')(x)
    x = layers.Conv2D(filters, kernel_size, padding='same',
                     activation='relu', name=f'{name_prefix}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name_prefix}_bn2')(x)
    return x


def build_attention_unet(input_shape=(64, 64, 3)):
    """
    Attention U-Net for Burn Detection

    Architecture:
    - Encoder: 4 downsampling blocks
    - Bottleneck: Deep feature extraction
    - Decoder: 4 upsampling blocks with attention gates
    - Skip connections: Enhanced with attention mechanism
    """
    inputs = layers.Input(input_shape, name='input')

    #  ENCODER
    # Block 1: 64x64 -> 32x32
    enc1 = conv_block(inputs, 32, name_prefix='enc1')
    pool1 = layers.MaxPooling2D(2, name='pool1')(enc1)

    # Block 2: 32x32 -> 16x16
    enc2 = conv_block(pool1, 64, name_prefix='enc2')
    pool2 = layers.MaxPooling2D(2, name='pool2')(enc2)

    # Block 3: 16x16 -> 8x8
    enc3 = conv_block(pool2, 128, name_prefix='enc3')
    pool3 = layers.MaxPooling2D(2, name='pool3')(enc3)

    # Block 4: 8x8 -> 4x4
    enc4 = conv_block(pool3, 256, name_prefix='enc4')
    pool4 = layers.MaxPooling2D(2, name='pool4')(enc4)

    #  BOTTLENECK: 4x4
    bottleneck = conv_block(pool4, 512, name_prefix='bottleneck')

    #  DECODER WITH ATTENTION
    # Block 1: 4x4 -> 8x8
    up1 = layers.UpSampling2D(2, name='up1')(bottleneck)
    up1 = layers.Conv2D(256, 2, padding='same', activation='relu', name='up1_conv')(up1)

    # Attention Gate 1
    att1 = AttentionGate(256, name='att_gate1')([up1, enc4])

    # Concatenate with attended skip connection
    merge1 = layers.Concatenate(name='merge1')([up1, att1])
    dec1 = conv_block(merge1, 256, name_prefix='dec1')

    # Block 2: 8x8 -> 16x16
    up2 = layers.UpSampling2D(2, name='up2')(dec1)
    up2 = layers.Conv2D(128, 2, padding='same', activation='relu', name='up2_conv')(up2)

    # Attention Gate 2
    att2 = AttentionGate(128, name='att_gate2')([up2, enc3])

    merge2 = layers.Concatenate(name='merge2')([up2, att2])
    dec2 = conv_block(merge2, 128, name_prefix='dec2')

    # Block 3: 16x16 -> 32x32
    up3 = layers.UpSampling2D(2, name='up3')(dec2)
    up3 = layers.Conv2D(64, 2, padding='same', activation='relu', name='up3_conv')(up3)

    # Attention Gate 3
    att3 = AttentionGate(64, name='att_gate3')([up3, enc2])

    merge3 = layers.Concatenate(name='merge3')([up3, att3])
    dec3 = conv_block(merge3, 64, name_prefix='dec3')

    # Block 4: 32x32 -> 64x64
    up4 = layers.UpSampling2D(2, name='up4')(dec3)
    up4 = layers.Conv2D(32, 2, padding='same', activation='relu', name='up4_conv')(up4)

    # Attention Gate 4
    att4 = AttentionGate(32, name='att_gate4')([up4, enc1])

    merge4 = layers.Concatenate(name='merge4')([up4, att4])
    dec4 = conv_block(merge4, 32, name_prefix='dec4')

    # OUTPUT
    outputs = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32', name='output')(dec4)

    model = models.Model(inputs=inputs, outputs=outputs, name='AttentionUNet_BurnDetection')
    return model


print("Building Attention U-Net model...")
model = build_attention_unet()
model.summary()

In [ ]:
# STEP 8: COMPILE MODEL
print("Compiling model...")

# Custom Dice Loss for better segmentation
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Dice coefficient for binary segmentation"""
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_loss(y_true, y_pred):
    """Dice loss (1 - Dice coefficient)"""
    return 1 - dice_coefficient(y_true, y_pred)

def combined_loss(y_true, y_pred):
    """Combined Dice + Binary Focal Loss for robust training"""
    focal = tf.keras.losses.BinaryFocalCrossentropy(
        alpha=0.25,
        gamma=2.0,
        from_logits=False
    )(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return 0.5 * focal + 0.5 * dice

# Compile with combined loss
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=1.0),
    loss=combined_loss,
    metrics=[
        'accuracy',
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.AUC(name='auc'),
        dice_coefficient
    ]
)

print("Model compiled with Attention mechanism.")

In [ ]:
# STEP 9: TRAINING WITH CALLBACKS
print("Starting training with Attention U-Net...\n")

callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'attention_unet_burn_best.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.TerminateOnNaN()  # Stop if NaN detected
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print("Training completed!")


In [ ]:
# STEP 10: EVALUATION
print("Evaluating Attention U-Net on test set...")

test_results = model.evaluate(X_test, y_test, verbose=0)
metric_names = ['Loss', 'Accuracy', 'Recall', 'Precision', 'AUC', 'Dice Coefficient']

print("FINAL RESULTS:")
print("=" * 60)
for name, value in zip(metric_names, test_results):
    print(f"  {name:20s}: {value:.4f}")
print("=" * 60)

In [ ]:
# STEP 11: VISUALIZATION
print("Generating graphs...\n")

# Plot training history
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history.history['accuracy'], label='Train Acc', linewidth=2)
axes[0, 1].plot(history.history['val_accuracy'], label='Val Acc', linewidth=2)
axes[0, 1].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Dice Coefficient
axes[0, 2].plot(history.history['dice_coefficient'], label='Train Dice', linewidth=2)
axes[0, 2].plot(history.history['val_dice_coefficient'], label='Val Dice', linewidth=2)
axes[0, 2].set_title('Dice Coefficient', fontsize=14, fontweight='bold')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Dice')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Recall
axes[1, 0].plot(history.history['recall'], label='Train Recall', linewidth=2)
axes[1, 0].plot(history.history['val_recall'], label='Val Recall', linewidth=2)
axes[1, 0].set_title('Model Recall', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Recall')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Precision
axes[1, 1].plot(history.history['precision'], label='Train Precision', linewidth=2)
axes[1, 1].plot(history.history['val_precision'], label='Val Precision', linewidth=2)
axes[1, 1].set_title('Model Precision', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Precision')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# AUC
axes[1, 2].plot(history.history['auc'], label='Train AUC', linewidth=2)
axes[1, 2].plot(history.history['val_auc'], label='Val AUC', linewidth=2)
axes[1, 2].set_title('Model AUC', fontsize=14, fontweight='bold')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_ylabel('AUC')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('attention_unet_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

# Visualize predictions on test set
print("Sample predictions with Attention U-Net:\n")

# Find patches with various burn levels
burn_levels = np.mean(y_test, axis=(1, 2, 3))
indices = [
    np.argmax(burn_levels),                          # Highest burn
    np.argsort(burn_levels)[-len(burn_levels)//4],  # High burn
    np.argsort(burn_levels)[len(burn_levels)//2],   #medium burn
    np.argmin(burn_levels)                           # Lowest burn
]

fig, axes = plt.subplots(len(indices), 4, figsize=(16, 4*len(indices)))

for i, idx in enumerate(indices):
    # Get prediction
    pred = model.predict(X_test[idx:idx+1], verbose=0)[0, :, :, 0]

    # RGB Input
    axes[i, 0].imshow(X_test[idx])
    axes[i, 0].set_title(f'RGB Input (Patch {idx})', fontweight='bold')
    axes[i, 0].axis('off')

    # Ground Truth
    im1 = axes[i, 1].imshow(y_test[idx, :, :, 0], cmap='Reds', vmin=0, vmax=1)
    axes[i, 1].set_title(f'Ground Truth ({100*burn_levels[idx]:.1f}% burned)', fontweight='bold')
    axes[i, 1].axis('off')
    plt.colorbar(im1, ax=axes[i, 1], fraction=0.046)

    # Prediction (continuous)
    im2 = axes[i, 2].imshow(pred, cmap='Reds', vmin=0, vmax=1)
    axes[i, 2].set_title(f'Attention Prediction ({100*np.mean(pred):.1f}% burned)', fontweight='bold')
    axes[i, 2].axis('off')
    plt.colorbar(im2, ax=axes[i, 2], fraction=0.046)

    # Binary prediction
    axes[i, 3].imshow(pred > 0.5, cmap='RdYlBu_r', vmin=0, vmax=1)
    axes[i, 3].set_title(f'Binary (>0.5 threshold)', fontweight='bold')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.savefig('attention_unet_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# STEP 12: VISUALIZE ATTENTION MAPS
print("Extracting attention maps...\n")

# Create a model to extract attention outputs
attention_layer_names = ['att_gate1', 'att_gate2', 'att_gate3', 'att_gate4']
attention_outputs = [model.get_layer(name).output for name in attention_layer_names]
attention_model = models.Model(inputs=model.input, outputs=attention_outputs)

# Get attention maps for a sample
sample_idx = np.argmax(burn_levels)
sample_input = X_test[sample_idx:sample_idx+1]
attention_maps = attention_model.predict(sample_input, verbose=0)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# Original image
axes[0].imshow(X_test[sample_idx])
axes[0].set_title('Input Image', fontweight='bold', fontsize=12)
axes[0].axis('off')

# Attention maps
titles = ['Attention 1\n(Low Level)', 'Attention 2\n(Mid Level)',
          'Attention 3\n(Mid-High Level)', 'Attention 4\n(High Level)']

for i, (att_map, title) in enumerate(zip(attention_maps, titles)):
    # Average across channels and resize to original size
    att_vis = np.mean(att_map[0], axis=-1)

    im = axes[i+1].imshow(att_vis, cmap='viridis', interpolation='bilinear')
    axes[i+1].set_title(title, fontweight='bold', fontsize=12)
    axes[i+1].axis('off')
    plt.colorbar(im, ax=axes[i+1], fraction=0.046)

plt.tight_layout()
plt.savefig('attention_maps_visualization.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("SUCCESS! ATTENTION U-NET TRAINING COMPLETE")
print(f"Model saved as: 'attention_unet_burn_best.h5'")
print(f"Training history plot: 'attention_unet_training_history.png'")
print(f"Prediction samples: 'attention_unet_predictions.png'")
print(f"Attention maps: 'attention_maps_visualization.png'")
print(f"\nFinal Performance:")
print(f"   • Test Accuracy:       {test_results[1]:.1%}")
print(f"   • Test Recall:         {test_results[2]:.1%}")
print(f"   • Test Precision:      {test_results[3]:.1%}")
print(f"   • Test AUC:            {test_results[4]:.4f}")
print(f"   • Test Dice Coeff:     {test_results[5]:.4f}")
print("Zero NaN errors encountered!")
print("Attention mechanism successfully integrated!")
